# FRED over MCP — from the category tree to a plotted series

FRED (Federal Reserve Economic Data, St. Louis Fed) republishes roughly 800,000
economic time series collected from about 100 sources: Treasury yields, CPI,
the national accounts, the labor market, and every regional cut of them.

Two things about FRED shape everything below.

**It is a tree, not a search index.** Every series hangs off a category, and the
root of that tree is `category_id` 0 with eight children. Depth is not uniform,
and a category can hold child categories *and* series of its own — so "descend
to a leaf" is the wrong mental model. Descend until a category answers
`fred_category_series` with something.

**A missing observation is a single period.** FRED writes `"."` where it has no
value for a date. meida's response models map that to `value: null` and keep the
row, so the series' calendar stays intact — which means anything that plots has
to skip those rows rather than cast them.

The MCP server must be running (`python -m mcp_server.server`); `MCP_URL`
selects it, defaulting to `http://localhost:8080/sse`.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from matplotlib import pyplot
from lib import config

from utils import (
    FRED_TOOL_PREFIXES,
    call_tool,
    unwrap,
    list_mcp_tools,
    show_tool_schema,
    plot_fred_series,
)

pyplot.style.use(config.glyfish_style)

## 1. Discovery — the FRED tools

One server carries every source, so the first question is which tools are FRED's.
Tool names are prefixed by source (`fred_`, `bls_`, `cdc_`, `tiingo_`, ...) with
a single exception: `list_releases` predates the convention. That is why
`FRED_TOOL_PREFIXES` is a pair rather than a string — filtering on `fred_` alone
silently drops a FRED tool.

The seven tools are three ways into the same catalog. `fred_category_*` walks the
topic tree; `list_releases` / `fred_release_series` walk publications instead
(everything issued in the *Employment Situation* release, say); and
`fred_series_updates` answers "what changed recently". This walkthrough takes the
category axis, because that is the one that starts from a question rather than
from an id you already have.

(Read `list_releases`' description below with suspicion: it is a copy of
`fred_release_series`'. The tool lists *releases* — the ids the other one takes.)

In [ ]:
await list_mcp_tools(prefix=FRED_TOOL_PREFIXES)

`fred_series_observations` is the tool that returns data, and its schema carries
the one default worth knowing before you call it: **`limit` is 100**. Ask for a
daily series spanning sixty years and you get its first hundred days, with no
error and nothing that looks wrong. `count` in the response is FRED's total for
the query, so `count != len(observations)` is how truncation announces itself —
check it, or page with `offset`.

`frequency` and `units` are server-side transforms: FRED will aggregate a daily
series to monthly, or difference it, before it crosses the wire, which beats
resampling a payload you did not need to move.

In [ ]:
schema = await show_tool_schema("fred_series_observations")


## 2. Finding a series

Someone who wants "the 10-year Treasury yield" does not begin with a series id —
they begin at the root and read their way down. Each id below was picked out of
the level printed above it: root → *Money, Banking, & Finance* → *Interest
Rates*, which is where the Treasury categories live.

`count` comes back `null` for categories — FRED reports no total for this
endpoint, so unlike a series listing, what you get is all there is.

In [ ]:
for category_id in (0, 32991, 22):
    children = unwrap(await call_tool("fred_category_children", {"category_id": category_id}))["categories"]
    print(f"\ncategory {category_id}: {len(children)} children")
    for child in children:
        print(f"  {child['id']:>6}  {child['name']}")


*Treasury Constant Maturity* (115) is where the descent stops: it answers with
series rather than more categories. FRED reports 63 of them and returns 63, so
nothing was truncated — the check that matters on every listing tool.

The tool exposes `order_by` but not `sort_order`, and FRED's default sort is
ascending, so `order_by="popularity"` would hand back the *least* used series
first. At 63 rows the honest fix is to sort what came back. Popularity is FRED's
0–100 usage score, and it is a decent proxy for "the series people mean when they
say this thing".

In [ ]:
payload = unwrap(await call_tool("fred_category_series", {"category_id": 115, "limit": 1000}))
series = sorted(payload["series"], key=lambda s: -(s["popularity"] or 0))
print(f"{payload['count']} series in the category, {len(payload['series'])} returned\n")

for entry in series[:8]:
    print(f"{entry['id']:8s} {entry['popularity']:>3}  {entry['frequency_short']:2s} "
          f"{entry['units_short']:2s} {entry['observation_start']} .. {entry['observation_end']}  "
          f"{entry['title'][:58]}")


## 3. Fetch and plot

`DGS10` is the constant-maturity 10-year Treasury yield — the price of a
risk-free long-dated dollar, and so the discount rate sitting underneath most
things priced in dollars. Daily since 1962, it carries the whole arc in one line:
the Great Inflation lifting it to 15.84% in September 1981, a forty-year descent
to 0.52% in the summer of 2020, and the snap back after 2021. Not many series
show a regime change this plainly.

Metadata first, so the plot can label itself from FRED's own title and units;
then the observations with `limit` raised, since the default 100 would not get
us out of 1962. The missing-value count is the `"."` sentinel made visible —
every market holiday since 1962, kept as a dated row with no value, and skipped
by the plot helper.

In [ ]:
series_id = "DGS10"

info = unwrap(await call_tool("fred_series_info", {"series_id": series_id}))["series"][0]
print(f"{info['title']}\n  {info['frequency']}, {info['units']}, {info['seasonal_adjustment']}"
      f"\n  {info['observation_start']} .. {info['observation_end']}, updated {info['last_updated']}")

observations = unwrap(await call_tool(
    "fred_series_observations", {"series_id": series_id, "limit": 20000}
))
missing = sum(1 for o in observations["observations"] if o["value"] is None)
print(f"\n{observations['count']} observations reported, "
      f"{len(observations['observations'])} returned, {missing} with no value")


In [ ]:
plot_fred_series(observations, info, figsize=(12, 6))